<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_J4_W5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install Required Libraries

In [ ]:
pip install gensim spacy torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.1 MB/s eta 0:00:00


## 2. Load and Preprocess the Dataset

In [14]:
import pandas as pd # Importation pour la manipulation de données
import numpy as np # Importation pour les calculs numériques
from sklearn.preprocessing import MinMaxScaler # Pour normaliser les données

# Fixer la graine aléatoire pour que les résultats soient identiques à chaque exécution
np.random.seed(42)
# Générer une séquence de 1000 dates consécutives
dates = pd.date_range(start='2020-01-01', periods=1000, freq='D')
# Simuler des prix boursiers avec une marche aléatoire cumulée
data = np.cumsum(np.random.randn(1000)) + 100
# Créer un tableau structuré (DataFrame) avec Date et Prix de clôture
df = pd.DataFrame({'Date': dates, 'Close': data})

# Créer la cible (Target) : c'est le prix du lendemain (on décale de 1 vers le haut)
df['Target'] = df['Close'].shift(-1)

# Supprimer la dernière ligne qui n'a plus de cible suite au décalage
df.dropna(inplace=True)

# Initialiser l'outil de mise à l'échelle entre 0 et 1
scaler = MinMaxScaler(feature_range=(0, 1))
# Appliquer la normalisation sur les colonnes prix actuel et prix cible
scaled_data = scaler.fit_transform(df[['Close', 'Target']])

# Afficher les premières lignes transformées et la taille totale
print(f"Aperçu des données normalisées :\n{scaled_data[:5]}")
print(f"Taille du dataset : {len(scaled_data)}")

Aperçu des données normalisées :
[[0.38783024 0.38470586]
 [0.38470586 0.39934175]
 [0.39934175 0.43375783]
 [0.43375783 0.42846664]
 [0.42846664 0.42317582]]
Taille du dataset : 999


## 3. Prepare the Dataset for Training

In [15]:
import torch # Importation de la bibliothèque principale PyTorch
from torch.utils.data import Dataset, DataLoader # Pour gérer les lots de données

# Définir les proportions : 80% entraînement, 10% validation, 10% test
train_size = int(len(scaled_data) * 0.8)
val_size = int(len(scaled_data) * 0.1)

# Découper le tableau de données en trois parties distinctes
train_data = scaled_data[:train_size]
val_data = scaled_data[train_size:train_size+val_size]
test_data = scaled_data[train_size+val_size:]

# Créer une classe pour transformer nos données en format compatible PyTorch
class StockDataset(Dataset):
    def __init__(self, data):
        # X contient le prix actuel, redimensionné pour le modèle LSTM (Batch, Seq, Feature)
        self.X = torch.tensor(data[:, 0], dtype=torch.float32).reshape(-1, 1, 1)
        # y contient le prix à prédire
        self.y = torch.tensor(data[:, 1], dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        # Indique à PyTorch la taille totale du dataset
        return len(self.X)

    def __getitem__(self, idx):
        # Permet de récupérer un exemple précis par son index
        return self.X[idx], self.y[idx]

# Créer des chargeurs de données qui gèrent le mélange (shuffle) et les lots (batchs)
train_loader = DataLoader(StockDataset(train_data), batch_size=32, shuffle=True)
val_loader = DataLoader(StockDataset(val_data), batch_size=32, shuffle=False)
test_loader = DataLoader(StockDataset(test_data), batch_size=32, shuffle=False)

print(f"Batches d'entraînement : {len(train_loader)}")

Batches d'entraînement : 25


## 4. Define the LSTM Model

In [1]:
import torch.nn as nn

class StockLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2, dropout=0.2):
        super(StockLSTM, self).__init__()
        # Couche LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        # Couche de Dropout
        self.dropout = nn.Dropout(dropout)
        # Couche linéaire (Dense) pour la sortie
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        out, (hn, cn) = self.lstm(x)
        # On prend la sortie du dernier pas de temps
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

# Initialisation du modèle
model = StockLSTM()
print(model)

StockLSTM(
  (lstm): LSTM(1, 50, num_layers=2, batch_first=True, dropout=0.2)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)


## 5. Train the Model

In [16]:
import torch
import torch.nn as nn # Modules de réseaux de neurones
import torch.optim as optim # Algorithmes d'optimisation

# Définition de l'architecture du modèle LSTM
class StockLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2, dropout=0.2):
        super(StockLSTM, self).__init__()
        # Couche LSTM pour capturer les dépendances temporelles
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        # Couche Dropout pour réduire le surapprentissage en désactivant des neurones
        self.dropout = nn.Dropout(dropout)
        # Couche linéaire pour transformer la sortie du LSTM en une seule valeur prédite
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Passage des données dans le LSTM
        out, _ = self.lstm(x)
        # On extrait seulement le dernier état de la séquence, on applique dropout et la couche finale
        out = self.fc(self.dropout(out[:, -1, :]))
        return out

# Créer une instance du modèle
model = StockLSTM()
# Définir la fonction de perte : Erreur Quadratique Moyenne (idéal pour la régression)
criterion = nn.MSELoss()
# Définir l'optimiseur Adam pour ajuster les poids du modèle
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Début de l'entraînement...")
for epoch in range(20): # Boucle sur 20 itérations complètes du dataset
    model.train() # Mettre le modèle en mode entraînement
    for inputs, targets in train_loader: # Parcourir les lots de données
        optimizer.zero_grad() # Effacer les anciens calculs de gradients
        outputs = model(inputs) # Faire une prédiction (Forward pass)
        loss = criterion(outputs, targets) # Calculer l'écart avec la réalité
        loss.backward() # Calculer les gradients (Backward pass)
        optimizer.step() # Mettre à jour les poids du modèle

    # Afficher la progression toutes les 5 époques
    if (epoch + 1) % 5 == 0:
        print(f'Époque [{epoch+1}/20], Perte: {loss.item():.6f}')

print("Entraînement terminé avec succès !")

Début de l'entraînement...
Époque [5/20], Perte: 0.014858
Époque [10/20], Perte: 0.001722
Époque [15/20], Perte: 0.001565
Époque [20/20], Perte: 0.001074
Entraînement terminé avec succès !


## 6. Evaluate the Model

In [17]:
from sklearn.metrics import r2_score # Pour mesurer la qualité de la prédiction
import joblib # Pour sauvegarder des objets Python comme le scaler

# Mettre le modèle en mode évaluation (désactive le dropout)
model.eval()
predictions = []
actuals = []

# Pas besoin de calculer les gradients pendant l'évaluation (gain de performance)
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs) # Prédire sur les données de test
        predictions.extend(outputs.numpy()) # Ajouter aux prédictions totales
        actuals.extend(targets.numpy()) # Ajouter aux valeurs réelles totales

# Calculer le score R² : plus il est proche de 1, meilleure est la prédiction
score = r2_score(actuals, predictions)
print(f"Score R² sur le jeu de test : {score:.4f}")

# Sauvegarder le scaler pour pouvoir transformer de futurs nouveaux prix
joblib.dump(scaler, 'scaler.gz')
# Sauvegarder les poids entraînés du modèle sur le disque
torch.save(model.state_dict(), 'stock_model.pth')

print("Modèle et scaler sauvegardés avec succès.")

Score R² sur le jeu de test : 0.0815
Modèle et scaler sauvegardés avec succès.
